In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps -q
!pip install unsloth_zoo --no-deps -q
!pip install xformers --no-deps -q
!pip install peft --no-deps -q
!pip install trl --no-deps -q
!pip install accelerate --no-deps -q
!pip install "tyro>=0.5.11" -q
!pip install bitsandbytes -q

# Verify
import torch
print(" PyTorch :", torch.__version__)
print(" CUDA    :", torch.version.cuda)
print(" GPU     :", torch.cuda.get_device_name(0))

import unsloth
print(" Unsloth : OK")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import gc
from unsloth import FastLanguageModel


ADAPTER_REPO   = "hagora-30/elara_report"
MERGED_REPO    = "hagora-30/Elara-14B-Merged"
MAX_SEQ_LENGTH = 4096


gc.collect()
torch.cuda.empty_cache()
print(f"GPU RAM free : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
print(f"GPU RAM total: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB\n")


print("Loading base model + LoRA adapters in 4-bit...")
print("(Takes 10–15 minutes on Colab T4)\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,     # CRITICAL for 12GB RAM
)

print(" Model loaded!\n")


gc.collect()
torch.cuda.empty_cache()


print("Merging and pushing to Hugging Face...")
print(f"Target: https://huggingface.co/{MERGED_REPO}")
print("(Takes 30–60 minutes — do NOT close Colab)\n")

model.push_to_hub_merged(
    repo_id        = MERGED_REPO,
    tokenizer      = tokenizer,
    save_method    = "merged_16bit",
    private        = True,
    token          = True,
    maximum_memory_usage = 0.7,   # Unsloth uses max 70% of available RAM
)

print("\n" + "="*60)
print(" DONE! Model pushed successfully!")
print(f"   https://huggingface.co/{MERGED_REPO}")
print("="*60)